**Part 5 · Projects** · step 22 of 25  
[← 21 Project: SQL agent](../21_sql_inventory_agent/sql_agent.ipynb) · [🏠 Roadmap](../../README.md) · [22 Project: dynamic tool agent (part 2) →](02_dynamic_tool_agent.ipynb)

# 22 · Project — Dynamic Tool Agent, Part 1: Database design

**Idea:** store agents *and their tools' source code* in a database, then build agents at runtime
(no redeploy to add a tool). This notebook sketches the full schema; Part 2 implements a
simplified version.

| Table | Holds |
|---|---|
| `agents` | name, model, system prompt, temperature |
| `tools` | tool name, description, **Python source code**, entry function, timeout |
| `agent_tools` | many-to-many link: which tools an agent can use (+ order) |
| `executions` | one row per agent run (input, output, status, latency) |
| `tool_executions` | one row per tool call inside a run |

Running this creates `sample.db` (schema only, no data). *Offline — no API key needed.*

In [ ]:
import sqlite3

conn = sqlite3.connect("sample.db")
cursor = conn.cursor()

In [ ]:
cursor.executescript("""
CREATE TABLE IF NOT EXISTS agents (
    id TEXT PRIMARY KEY,
    name TEXT NOT NULL UNIQUE,
    description TEXT,
    model TEXT NOT NULL,
    system_prompt TEXT,
    temperature REAL DEFAULT 0.0,
    is_active INTEGER DEFAULT 1,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP,
    updated_at DATETIME DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS tools (
    id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    description TEXT,
    type TEXT NOT NULL DEFAULT 'script',
    code TEXT NOT NULL,                       -- Python source of the tool
    entry_function TEXT DEFAULT 'execute',    -- function inside `code` to call
    requirements TEXT DEFAULT '[]',           -- JSON list of pip packages
    timeout_seconds INTEGER DEFAULT 30,
    version INTEGER DEFAULT 1,
    is_active INTEGER DEFAULT 1,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP,
    updated_at DATETIME DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS agent_tools (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    agent_id TEXT NOT NULL,
    tool_id TEXT NOT NULL,
    execution_order INTEGER DEFAULT 0,
    FOREIGN KEY(agent_id) REFERENCES agents(id),
    FOREIGN KEY(tool_id) REFERENCES tools(id)
);

CREATE TABLE IF NOT EXISTS executions (
    id TEXT PRIMARY KEY,
    agent_id TEXT,
    input TEXT,
    output TEXT,
    status TEXT,
    execution_time_ms INTEGER,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS tool_executions (
    id TEXT PRIMARY KEY,
    execution_id TEXT,
    tool_id TEXT,
    tool_input TEXT,
    tool_output TEXT,
    status TEXT,
    execution_time_ms INTEGER
);
""")
conn.commit()

[row[0] for row in cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")]

In [ ]:
conn.close()

---
✅ **Done with 22 Project: dynamic tool agent (part 1).** Next: [22 Project: dynamic tool agent (part 2) →](02_dynamic_tool_agent.ipynb)  
[🏠 Roadmap](../../README.md)